In [ ]:
# ============================================================
# CONFIG — edit only this cell
# ============================================================
from pathlib import Path
from mdatools.config import AnalysisConfig

cfg = AnalysisConfig(
    ligand_resname  = "LIG",
    topology_glob   = "*.pdb",
    trajectory_glob = "*.xtc",
    dt_ns           = 2.0,
)

REPLICA_ROOTS = [
    Path("../run01"),
    Path("../run02"),
    Path("../run03"),
]

SPLIT_FRAC    = 0.5   # fraction for first/second-half comparison
MAX_BLOCK_SIZE = None  # None = auto (len // 4)
MAX_LAG        = 500   # frames for autocorrelation time

OUTPUT_DIR = Path("./figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# ============================================================

## Step 1 — Run RMSD analysis for all replicas

In [ ]:
from mdatools.analysis.rmsd import run_rmsd_batch

rmsd_results = run_rmsd_batch(REPLICA_ROOTS, cfg)
for name, res in rmsd_results.items():
    print(f"{name}: {len(res.df)} frames, backbone RMSD mean={res.df['Backbone'].mean():.2f} Å")

## Step 2 — RMSD convergence (first vs second half)

In [ ]:
from mdatools.analysis.convergence import ConvergenceAnalyzer
from mdatools.plotting.convergence_plots import plot_rmsd_convergence

analyzer = ConvergenceAnalyzer(cfg)
conv_results = {}

for name, res in rmsd_results.items():
    conv = analyzer.rmsd_convergence(res.df, split_frac=SPLIT_FRAC)
    conv_results[name] = conv
    print(f"{name}: {conv.recommendation}")
    fig = plot_rmsd_convergence(
        conv, res.df,
        save_path=OUTPUT_DIR / f"convergence_{name}.png"
    )
    display(fig)

## Step 3 — Block error analysis

In [ ]:
from mdatools.plotting.convergence_plots import plot_block_error

for name, res in rmsd_results.items():
    block_df = analyzer.block_error(res.df["Backbone"], max_block_size=MAX_BLOCK_SIZE)
    fig = plot_block_error(
        block_df,
        observable_name=f"Backbone RMSD ({name})",
        save_path=OUTPUT_DIR / f"block_error_{name}.png",
    )
    display(fig)

## Step 4 — Autocorrelation time

In [ ]:
for name, res in rmsd_results.items():
    tau = analyzer.autocorrelation_time(res.df["Backbone"], max_lag=MAX_LAG)
    tau_ns = tau * cfg.dt_ns
    print(f"{name}: τ_int ≈ {tau:.1f} frames ({tau_ns:.1f} ns)")

## Step 5 — Replica consistency

In [ ]:
from mdatools.plotting.convergence_plots import plot_replica_overlap

overlap_df = analyzer.replica_consistency(list(rmsd_results.values()))
print(overlap_df.to_string(index=False))

fig = plot_replica_overlap(
    overlap_df,
    save_path=OUTPUT_DIR / "replica_overlap.png",
)
fig